# Transformation Crystallography End To End

Five questions come up again and again when a phase transformation is studied by
EBSD and TEM, and this notebook answers all five on one system — the **Burgers**
$\beta \rightarrow \alpha$ relationship in titanium, the reconstructive
bcc$\rightarrow$hcp transformation that also governs zirconium and hafnium.

1. I measured Euler angles for a parent grain and a child grain. **What is the
   orientation relationship?**
2. Given the relationship, **what does an arbitrary parent plane or direction
   become in each of the product variants?**
3. Given a parent zone axis, **what does the composite diffraction pattern look
   like**, and can I get it out as a figure *and* a table?
4. What if I instead **tilt onto a zone of the product**?
5. Here is a **measured pattern** — **solve it.**

Each section states the crystallography first and then computes it, so the code
is a check on the physics rather than a substitute for it. Every number printed
below is computed live when this notebook runs.

The Burgers relationship is
$$(011)_{\beta} \parallel (0001)_{\alpha}, \qquad
[\bar{1}11]_{\beta} \parallel [\bar{1}2\bar{1}0]_{\alpha},$$
with **12 variants**: the six $\{110\}_{\beta}$ planes that can become the basal
plane, times the two $\langle 111 \rangle_{\beta}$ directions in each that can
become $\langle 11\bar{2}0 \rangle_{\alpha}$.

In [ ]:
import json
import tempfile
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning, module="pymatgen.*")

from pytex import (
    AtomicSite,
    CrystalDirection,
    CrystalPlane,
    FrameDomain,
    Handedness,
    Lattice,
    MillerIndex,
    OrientationRelationship,
    OrientationSet,
    Phase,
    ReferenceFrame,
    Rotation,
    SymmetrySpec,
    UnitCell,
    ZoneAxis,
    characterize_orientation_relationship,
    describe_orientation_relationship,
    orientation_relationship_from_euler,
    specimen_frame,
    variant_correspondence_table,
)
from pytex.diffraction import (
    KinematicSimulationConfig,
    MeasuredSAEDPattern,
    MeasuredSpot,
    PatternCalibration,
    assign_transformation_variant,
    composite_reflection_table,
    export_composite_saed,
    simulate_composite_saed,
    simulate_composite_saed_from_child_zone,
    solve_saed_pattern,
)
from pytex.plotting.composite_saed import CompositeSAEDPlotConfig, render_composite_saed

np.set_printoptions(precision=6, suppress=True)
plt.rcParams["figure.dpi"] = 110

## 0. The two phases

$\beta$-Ti is body-centred cubic ($Im\bar{3}m$) and $\alpha$-Ti is hexagonal
close-packed ($P6_3/mmc$). **Declaring the space groups matters**: PyTex reads
the lattice centring from the first letter of the symbol, and a phase supplied
without one is simulated as *primitive* — so a bcc phase without its symbol
would show $\{100\}$ reflections that the real structure forbids. Section 3
checks this explicitly.

In [ ]:
beta_frame = ReferenceFrame("beta_crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT)
alpha_frame = ReferenceFrame("alpha_crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT)

A_BETA = 3.3065
A_ALPHA, C_ALPHA = 2.9508, 4.6855

beta_lattice = Lattice(A_BETA, A_BETA, A_BETA, 90.0, 90.0, 90.0, crystal_frame=beta_frame)
alpha_lattice = Lattice(A_ALPHA, A_ALPHA, C_ALPHA, 90.0, 90.0, 120.0, crystal_frame=alpha_frame)

beta_ti = Phase(
    "beta-titanium",
    lattice=beta_lattice,
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=beta_frame),
    crystal_frame=beta_frame,
    unit_cell=UnitCell(
        lattice=beta_lattice,
        sites=(
            AtomicSite(label="Ti1", species="Ti", fractional_coordinates=np.array([0.0, 0.0, 0.0])),
            AtomicSite(label="Ti2", species="Ti", fractional_coordinates=np.array([0.5, 0.5, 0.5])),
        ),
    ),
    space_group_symbol="Im-3m",
)
alpha_ti = Phase(
    "alpha-titanium",
    lattice=alpha_lattice,
    symmetry=SymmetrySpec.from_point_group("6/mmm", reference_frame=alpha_frame),
    crystal_frame=alpha_frame,
    unit_cell=UnitCell(
        lattice=alpha_lattice,
        sites=(
            AtomicSite(label="Ti1", species="Ti", fractional_coordinates=np.array([0.0, 0.0, 0.0])),
            AtomicSite(
                label="Ti2", species="Ti",
                fractional_coordinates=np.array([1.0 / 3.0, 2.0 / 3.0, 0.5]),
            ),
        ),
    ),
    space_group_symbol="P6_3/mmc",
)

burgers = OrientationRelationship.from_burgers_correspondence(
    parent_phase=beta_ti, child_phase=alpha_ti
)
variants = burgers.generate_variants()
print(f"{len(variants)} Burgers variants")
print(f"misorientation representative: {burgers.misorientation().angle_deg:.3f} deg")

## 1. What is the orientation relationship?

The measurement is two columns of Euler angles: one parent grain, several child
grains. To have ground truth, the "measurement" here is synthesized from six
known variants of one parent and then **thrown away** — the analysis is given
nothing but the angles.

`orientation_relationship_from_euler` fits the operative rotation *without* being
told which relationship to expect. The starting estimate comes from the data:
one pair is reduced to its minimum-angle representative in the double coset
$G_c \mathbf{V}_0 G_p$, which absorbs the parent symmetry operation that
distinguishes one variant from another, so pairs from different variants still
align to it.

In [ ]:
rng = np.random.default_rng(20260804)
frame = specimen_frame()

parent_rotation = Rotation.from_axis_angle([1.0, 2.0, 3.0], 0.7)
P = parent_rotation.as_matrix()
chosen = (1, 3, 5, 7, 9, 11)

# Canonical crystal->specimen convention: C = P V^T.
child_matrices = np.stack(
    [P @ variants[k - 1].parent_to_child_rotation.as_matrix().T for k in chosen]
)
parents = OrientationSet.from_matrices(
    np.stack([P] * len(chosen)), specimen_frame=frame, phase=beta_ti
)
children = OrientationSet.from_matrices(child_matrices, specimen_frame=frame, phase=alpha_ti)

parent_euler = parents.as_bunge_euler(degrees=True)
child_euler = children.as_bunge_euler(degrees=True)
print("Parent Euler angles (Bunge, deg):")
print(parent_euler[0])
print("\nChild Euler angles (Bunge, deg), one row per measured child grain:")
print(child_euler)

In [ ]:
report = orientation_relationship_from_euler(
    parent_euler, child_euler, parent_phase=beta_ti, child_phase=alpha_ti
)
print(report.describe())

Read that output carefully, because it contains four separable claims.

- The **fitted rotation** and the **pair scatter** about it. Zero scatter here
  because the data are exact; on real measurements this is the number that says
  whether the pairs agree with each other at all.
- The **named match** and its margin over the runner-up. Burgers leads
  Shoji-Nishiyama by tens of degrees, which is why the verdict is confident.
- The **crystallographic statement** — the parallel planes and directions,
  which is how the literature reports an orientation relationship. Note the
  hexagonal side is written in four-index Miller-Bravais form.
- The **verdict**. `is_conclusive` is deliberately conservative: it requires the
  winner both to fit within tolerance *and* to lead the runner-up by more than
  the measurement scatter and its own misfit.

Adding realistic EBSD scatter shows the fit doing its job — averaging noise
that a single pair could not.

In [ ]:
def perturb(matrix, sigma_deg, generator):
    axis = generator.normal(size=3)
    axis /= np.linalg.norm(axis)
    angle = np.radians(generator.normal(0.0, sigma_deg))
    return Rotation.from_axis_angle(axis, angle).as_matrix() @ matrix


for sigma in (0.0, 0.5, 2.0):
    noisy = np.stack([perturb(m, sigma, rng) for m in child_matrices])
    noisy_children = OrientationSet.from_matrices(noisy, specimen_frame=frame, phase=alpha_ti)
    noisy_report = characterize_orientation_relationship(parents, noisy_children)
    print(
        f"scatter {sigma:.1f} deg -> {noisy_report.best_catalog_name:20s} "
        f"deviation {noisy_report.best_catalog_deviation_deg:5.3f} deg, "
        f"pair scatter {noisy_report.mean_residual_deg:5.3f} deg, "
        f"conclusive {noisy_report.is_conclusive}"
    )

The fitted rotation stays far closer to Burgers than the individual pairs do:
that gap *is* the averaging.

The statement can also be read straight out of any relationship, measured or
named — a rotation matrix is unreadable, but "$(011)_{\beta}$ parallel to
$(0001)_{\alpha}$" is the working crystallographic fact.

A rotation typically satisfies *several* exact low-index parallelisms at once,
all of them true. Which one the literature quotes depends on the two structures'
close-packed planes and directions, which a rotation does not know, so the
search prefers the relationship's own recorded defining families.

In [ ]:
planes, directions = describe_orientation_relationship(burgers)
print("Plane parallelisms:")
for statement in planes:
    print(f"  {statement.as_text():34s}  deviation {statement.deviation_deg:.4f} deg")
print("\nDirection parallelisms:")
for statement in directions:
    print(f"  {statement.as_text():34s}  deviation {statement.deviation_deg:.4f} deg")

## 2. What does a parent plane become in every variant?

A rotation has three degrees of freedom, so one plane parallelism plus one
in-plane direction parallelism determines it completely. Everything else follows
— including what any *other* plane or direction becomes, in each of the 12
variants.

The table below groups variants whose images are crystallographically
equivalent. Burgers has **six packets of two**: each of the six $\{110\}_{\beta}$
planes becomes the basal plane in exactly two variants, so a nominated member —
here $(011)$ — is basal in exactly two of the twelve.

In [ ]:
plane_110 = CrystalPlane(MillerIndex(np.array([0, 1, 1]), phase=beta_ti), phase=beta_ti)
table = variant_correspondence_table(burgers, plane_110)
print(table.describe())
print()
print(table.to_markdown())

The exactly-parallel rows are the physically interesting subset: those two
variants are the ones whose basal plane *is* this $(011)_{\beta}$. The remaining
ten land on irrational images, reported as the nearest low-index plane with an
honest angular residual.

Exact correspondences are a property of the relationship and do not depend on
the rationalization bound; how the *irrational* images are grouped does, because
a larger bound splits them across more index families. `describe()` says so
rather than letting "six distinct images" be read as physics when it is partly a
bookkeeping choice.

The reverse map is not selective in the same way. Every variant's basal plane
came from *some* $\{110\}_{\beta}$, so mapping $(0001)_{\alpha}$ back lands on a
$\{110\}_{\beta}$ plane in **all twelve**.

In [ ]:
basal = CrystalPlane(MillerIndex(np.array([0, 0, 1]), phase=alpha_ti), phase=alpha_ti)
reverse = variant_correspondence_table(burgers, basal, sense="child_to_parent")
print(reverse.describe())

direction_table = variant_correspondence_table(
    burgers, CrystalDirection([-1.0, 1.0, 1.0], phase=beta_ti)
)
print()
print(direction_table.describe())

## 3. The composite diffraction pattern from a parent zone axis

Down $[110]_{\beta}$ the beam looks straight along the $c$-axis of every variant
whose basal plane is that particular $\{110\}$ — the defining plane parallelism,
seen directly. Those variants show the six-fold basal pattern; the others are
off their own zone and contribute fewer, weaker reflections.

All sub-patterns share one **parent-anchored** detector basis, so their detector
coordinates overlay a single detector exactly as they would on the microscope's
screen.

In [ ]:
config = KinematicSimulationConfig(
    beam_energy_kev=200.0, camera_constant_mm_angstrom=180.0, max_index=6
)
zone_beta = ZoneAxis(np.array([1, 1, 0]), phase=beta_ti)
composite = simulate_composite_saed(
    burgers, zone_beta, variant_indices=(1, 2, 3, 4), config=config
)
print(composite.describe())

Two things in that report are worth pausing on.

**The centring audit.** Both phases declare a space group, so their lattice
centring is *declared* rather than assumed. Had `beta_ti` been built without
`space_group_symbol="Im-3m"`, the audit would say `ASSUMED` and the pattern
would carry $\{100\}_{\beta}$ reflections that body centring forbids. This is a
silent failure mode, so the library refuses to keep it silent:

In [ ]:
print("phase            centring  declared")
for name, centring, declared in composite.centering_audit():
    print(f"{name:16s} {centring:8s}  {declared}")

parent_spots = composite.parent_spots
odd = sum(1 for row in parent_spots.hkl if int(row.sum()) % 2 != 0)
print(f"\nbeta reflections with h+k+l odd (forbidden by I centring): {odd}")

**Intensities are normalized within each sub-pattern separately.** Kinematic
theory defines no intensity ratio between two different phases, so comparing a
$\beta$ spot's intensity with an $\alpha$ spot's is meaningless. Compare within
one source only.

Now the figure and the table — the two things a pattern has to become before it
can leave the notebook.

In [ ]:
figure = render_composite_saed(
    composite,
    config=CompositeSAEDPlotConfig(
        figsize=(7.0, 7.0),
        title=r"Burgers composite SAED, $\beta$ [110] zone",
    ),
)
plt.show()

In [ ]:
reflections = composite_reflection_table(composite)
print(reflections.describe())
print()
print(reflections.to_markdown(max_rows=12))

Every value in that table is read from the arrays the engine produced, so the
table and the figure cannot disagree. Note the detector radius uses the
**in-plane** part of $\mathbf{g}$: the difference from $(L\lambda)|\mathbf{g}|$
is the out-of-plane component that the excitation error records, while
$d = 1/|g|$ uses the full vector.

`export_composite_saed` writes the table, the figure, the parent/child
coincidence table and a JSON manifest — the relationship, both phases with their
applied centring, every variant's exact and nearest-rational child zone axis,
and the full simulation configuration — so the result is reproducible without
this notebook.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    export = export_composite_saed(composite, directory, stem="burgers_110")
    print(export.describe())
    print()
    manifest = json.loads(export.manifest_path.read_text(encoding="utf-8"))
    for key in ("relationship", "parent_zone_axis_label", "total_reflection_count",
                "intensity_normalization", "theory_level"):
        print(f"{key:28s} {manifest[key]}")

## 4. Anchoring on a product zone instead

Section 3 matches how the crystallography is derived, but not how the microscope
is used. In practice the operator tilts onto a low-index zone of the **product**
— say $[0001]_{\alpha}$ of one variant — and then wants to know what the matrix
and the sibling variants contribute to that same pattern.

The anchor variant's rotation $\mathbf{R}_k$ carries parent Cartesian vectors
into that child's frame, so the requested child zone corresponds to the parent
direction $\mathbf{R}_k^{\mathsf{T}}\mathbf{z}_c$ — **generally irrational**,
and reported exactly alongside its nearest rational label, exactly the honesty
child zone axes already receive.

In [ ]:
child_anchored = simulate_composite_saed_from_child_zone(
    burgers,
    ZoneAxis(np.array([0, 0, 1]), phase=alpha_ti),
    anchor_variant_index=3,
    variant_indices=(1, 2, 3, 4),
    config=config,
)
print("Anchored on:", child_anchored.anchor_description())
print("Parent zone axis:", child_anchored.parent_zone_axis_label())

Because the geometry is built through the *same* call the parent-anchored path
uses, there is one detector-geometry definition and a testable identity:
anchoring on variant $k$'s image of a parent zone reproduces the parent-anchored
pattern for that zone exactly.

In [ ]:
recovered = simulate_composite_saed_from_child_zone(
    burgers,
    composite.variant_pattern(3).zone_axis_child,
    anchor_variant_index=3,
    variant_indices=(1, 2, 3, 4),
    config=config,
)
largest = max(
    float(np.max(np.abs(
        composite.variant_pattern(k).spots.detector_mm
        - recovered.variant_pattern(k).spots.detector_mm
    )))
    for k in (1, 2, 3, 4)
)
parent_shift = float(np.max(np.abs(
    composite.parent_spots.detector_mm - recovered.parent_spots.detector_mm
)))
print(f"largest variant spot displacement: {largest:.2e} mm")
print(f"parent spot displacement:          {parent_shift:.2e} mm")

In [ ]:
figure = render_composite_saed(
    child_anchored,
    config=CompositeSAEDPlotConfig(
        figsize=(7.0, 7.0),
        title=r"Composite anchored on $[0001]_{\alpha}$ of variant 3",
    ),
)
plt.show()

## 5. Solving a measured pattern

Finally, the inverse problem. A measured pattern is a list of spot positions
relative to the transmitted beam, plus enough calibration to turn them into
reciprocal-space lengths. Nothing else is needed, and nothing else is used —
**intensities are never used for indexing**, because a kinematic intensity model
is not reliable enough to index against and a printed pattern rarely carries
calibrated intensities at all.

To have ground truth, the "measurement" is again the simulation's spot positions
handed back as if a user had clicked them.

In [ ]:
anchored_variant = child_anchored.variant_pattern(3)
measured = MeasuredSAEDPattern(
    name="alpha_basal_zone",
    spots=tuple(
        MeasuredSpot(position=(float(x), float(y)))
        for x, y in anchored_variant.spots.detector_mm
    ),
    calibration=PatternCalibration(units="mm", camera_constant_mm_angstrom=180.0),
)
print(measured.describe())

In [ ]:
solution_report = solve_saed_pattern(measured, [alpha_ti, beta_ti], max_index=6)
print(solution_report.describe())

The solver was offered both phases and picked the right one, down the right
zone, indexing every spot. Three properties of that report matter more than the
answer itself.

**The zone-sense ambiguity is intrinsic.** A single SAED pattern cannot
distinguish a zone axis from its reverse when the reflection set is
centrosymmetric, because inverting the crystal leaves the pattern unchanged. The
report names this rather than presenting one sense as *the* answer.

**Symmetry-equivalent descriptions are one answer, not several.** They are
deduplicated under the crystal point group and the survivor is rewritten into
the conventional description, so an unambiguous solve does not look contested.

**Not solving is a legitimate outcome.** Offered only the wrong phase, the
solver does not produce a plausible-looking wrong answer:

In [ ]:
wrong_phase = solve_saed_pattern(measured, [beta_ti], max_index=6)
if wrong_phase.solutions:
    best_wrong = wrong_phase.best()
    print(f"beta-only: best explains {100 * best_wrong.matched_fraction:.0f}% of the spots")
else:
    print("beta-only: no solution at all")
print("conclusive:", wrong_phase.is_conclusive)

And once the pattern is solved, the variant follows. The shared detector basis
has columns $(u, v, z)$ in *parent crystal* Cartesian coordinates, so a parent
vector's pattern coordinates are $\mathbf{B}^{\mathsf{T}}\mathbf{p}$ — the
parent's crystal-to-pattern rotation is $\mathbf{B}^{\mathsf{T}}$. Given that,
the child orientation each variant predicts is
$\mathbf{P}\mathbf{V}_k^{\mathsf{T}}$, and the closest prediction wins.

In [ ]:
parent_orientation = Rotation.from_matrix(
    np.ascontiguousarray(child_anchored.zone_basis_parent.T)
)
assigned = assign_transformation_variant(
    solution_report.best(), burgers, parent_orientation
)
print(f"planted variant:   3")
print(f"assigned variant:  {assigned.variant_index}")
print(f"deviation:         {assigned.variant_deviation_deg:.4f} deg")

### Picking the spots yourself

The measured pattern above was built in code. In practice the spots are clicked:

```python
from pytex.plotting.saed_picker import SAEDSpotPicker

picker = SAEDSpotPicker(image, calibration=calibration).show()
# left click adds, right click removes the nearest, middle click sets the beam
# centre, "u" undoes, "c" clears
picker.save_yaml("alpha_zone_01.yaml", name="alpha_zone_01")
```

and then solved from the saved file with `solve_saed_pattern_file`. The **YAML
file, not the click session, is the reproducibility boundary**: a pattern solved
from a committed file gives the same answer on any machine. The picking logic
itself lives in `SpotPickerState`, a plain object with no Matplotlib dependency,
so it is tested headlessly rather than being untestable GUI code.

Here is what such a file looks like — written and read back to show the round
trip is exact.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    path = measured.to_yaml(Path(directory) / "alpha_basal_zone.yaml")
    text = path.read_text(encoding="utf-8")
    print("\n".join(text.splitlines()[:14]))
    print("...")
    restored = MeasuredSAEDPattern.from_yaml(path)
    difference = float(np.max(np.abs(
        restored.g_vectors_inv_angstrom() - measured.g_vectors_inv_angstrom()
    )))
    print(f"\nround-trip difference: {difference:.1e} 1/angstrom")

## What was and was not shown

Everything above is **kinematic**: no dynamical (Bloch-wave) intensities, no
double diffraction, no HOLZ rings. The intensities rank reflections; they do not
predict what a plate of a given thickness will actually show.

The solver assumes the spots form a zone-axis pattern about a low-index zone. A
crystal tilted *off* zone — a variant seen from a parent zone axis, whose own
child zone is irrational — will be only partly indexed, and that partial match
is the honest outcome rather than a bug to work around.

The orientation-relationship determination in section 1 is validated
synthetically. Measured-EBSD fixtures remain outstanding, and no claim of
agreement with MTEX is made anywhere in this notebook.

### Where to go next

- {doc}`../../workflows/composite_or_diffraction` — the simulation workflow,
  including the lattice-centring trap in full
- {doc}`../../workflows/saed_pattern_solving` — the solving workflow
- {doc}`../../concepts/orientation_relationships` — the concept page
- {doc}`22_burgers_beta_to_alpha_zirconium` — the same relationship as a
  single-system deep dive, on the alloy Burgers himself worked on